In [ ]:
import pandas as pd

homes = pd.read_csv("../data/homes_clean.csv", parse_dates=["sold"])
homes.shape

In [ ]:
homes.groupby("area")["price"].median().sort_values()

In [ ]:
homes["per_sqft"] = homes["price"] / homes["sqft"]
homes.groupby("area").agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    median_per_sqft=("per_sqft", "median"),
).round()

In [ ]:
homes.pivot_table(
    index="area", columns="beds", values="price", aggfunc="median"
).round(-3)

In [ ]:
areas = pd.read_csv("../data/areas.csv")
areas

In [ ]:
check = homes.merge(
    areas, on="area", how="left", validate="many_to_one", indicator=True
)
check["_merge"].value_counts()

In [ ]:
check.loc[check["_merge"] == "left_only", "area"].unique()

In [ ]:
hows = ["inner", "left", "right", "outer"]
{how: len(homes.merge(areas, on="area", how=how)) for how in hows}

In [ ]:
homes["area"] = homes["area"].str.title()
homes = homes.merge(areas, on="area", validate="many_to_one")
homes.shape

In [ ]:
quarterly = homes.set_index("sold")["price"].resample("QS").median()
smooth = quarterly.rolling(4).mean()
trend = pd.DataFrame({"median": quarterly, "rolling_4q": smooth})
trend.round(-3)

In [ ]:
first, last = trend["rolling_4q"].dropna().iloc[[0, -1]]
round(last / first - 1, 3)